# Tensor Subclassing — __torch_function__ & __torch_dispatch__

**What you'll learn:**
- What dispatch protocols are and why they exist
- `__torch_function__`: intercept operations at the Python API level
- `TorchFunctionMode`: scoped operation interception without subclassing
- `__torch_dispatch__`: intercept at the ATen operator level
- `TorchDispatchMode`: scoped ATen-level interception
- Comparing what each level "sees" for the same operation
- Practical examples: scaled tensors, operation logging, unit tensors

**Prerequisites:** Comfortable with tensors, `nn.Module`, and Python `__magic__` methods.

**All code runs on CPU.**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.overrides import TorchFunctionMode
from torch.utils._python_dispatch import TorchDispatchMode
from collections import defaultdict

print(f"PyTorch version: {torch.__version__}")

## What Are Dispatch Protocols?

When you call `torch.add(a, b)`, PyTorch doesn't immediately run the operation. Instead, it goes through a **dispatch stack**:

```
User calls:  torch.add(a, b)
     ↓
[__torch_function__]  ← Python-level override (sees torch.add)
     ↓
[Autograd]            ← Records operations for backward
     ↓
[__torch_dispatch__]  ← ATen-level override (sees aten::add.Tensor)
     ↓
[Backend]             ← CPU/CUDA kernel actually runs
```

**Why this matters:** You can intercept operations at either level to:
- Add logging/profiling
- Transform tensor data (quantization, sparsity)
- Enforce constraints (type checking, shape validation)
- Build custom tensor types with new semantics

## __torch_function__: Python-Level Override

Define `__torch_function__` on a tensor subclass to intercept ALL PyTorch operations at the Python API level. Let's create a `ScaledTensor` that stores a scale factor separately.

In [ ]:
class ScaledTensor(torch.Tensor):
    """A tensor that carries a scalar scale factor. Actual data = tensor_data * scale."""

    @staticmethod
    def __new__(cls, data, scale=1.0):
        # Wrap existing tensor data
        return torch.Tensor._make_subclass(cls, data)

    def __init__(self, data, scale=1.0):
        self._scale = scale

    @property
    def scale(self):
        return self._scale

    def unscaled(self):
        """Get the true values (data * scale)."""
        return torch.Tensor._make_subclass(torch.Tensor, self) * self._scale

    @classmethod
    def __torch_function__(cls, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}

        # Unwrap ScaledTensors for the actual computation
        def unwrap(t):
            if isinstance(t, ScaledTensor):
                return torch.Tensor._make_subclass(torch.Tensor, t)
            return t

        unwrapped_args = torch.utils._pytree.tree_map(unwrap, args)
        unwrapped_kwargs = torch.utils._pytree.tree_map(unwrap, kwargs)

        # Run the operation on raw data
        result = func(*unwrapped_args, **unwrapped_kwargs)

        # Re-wrap if result is a tensor (propagate scale from first ScaledTensor arg)
        if isinstance(result, torch.Tensor):
            scale = next((a._scale for a in torch.utils._pytree.tree_leaves(args)
                         if isinstance(a, ScaledTensor)), 1.0)
            return ScaledTensor(result, scale=scale)
        return result

    def __repr__(self):
        return f"ScaledTensor(data={torch.Tensor._make_subclass(torch.Tensor, self)}, scale={self._scale})"

# Test it
st = ScaledTensor(torch.tensor([1.0, 2.0, 3.0]), scale=0.5)
print(f"Created: {st}")
print(f"True values: {st.unscaled()}")

## Testing ScaledTensor with PyTorch Operations

The `__torch_function__` protocol means standard PyTorch ops automatically work with our custom tensor.

In [ ]:
a = ScaledTensor(torch.tensor([1.0, 2.0, 3.0]), scale=2.0)
b = ScaledTensor(torch.tensor([4.0, 5.0, 6.0]), scale=2.0)

# All standard ops go through __torch_function__
result_add = torch.add(a, b)
result_mul = torch.mul(a, b)
result_relu = torch.relu(a - 5)

print(f"a = {a}")
print(f"b = {b}")
print(f"torch.add(a, b) = {result_add}")
print(f"  → type: {type(result_add).__name__}, scale preserved: {result_add.scale}")
print(f"torch.mul(a, b) = {result_mul}")
print(f"torch.relu(a - 5) = {result_relu}")
print(f"\nAll operations return ScaledTensors — dispatch protocol works!")

## TorchFunctionMode: Scoped Interception Without Subclassing

Sometimes you want to intercept operations in a **scope** (like logging all ops during a forward pass) without creating a tensor subclass. `TorchFunctionMode` provides this.

In [ ]:
class LoggingMode(TorchFunctionMode):
    """Log all torch operations within a scope."""

    def __init__(self):
        super().__init__()
        self.ops = []

    def __torch_function__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        self.ops.append(func.__name__ if hasattr(func, '__name__') else str(func))
        return func(*args, **kwargs)

# Use as a context manager — only active within the `with` block
x = torch.randn(4, 8)

with LoggingMode() as mode:
    y = x @ x.T
    z = torch.relu(y)
    w = z.softmax(dim=-1)

print(f"Operations captured in scope:")
for i, op in enumerate(mode.ops, 1):
    print(f"  {i}. {op}")
print(f"\nTotal: {len(mode.ops)} operations")

## TorchFunctionMode: Operation Counter on a Model Forward Pass

A practical use: count operations during a model's forward pass for profiling.

In [ ]:
class OpCounter(TorchFunctionMode):
    """Count operations by category during a forward pass."""

    def __init__(self):
        super().__init__()
        self.counts = defaultdict(int)

    def __torch_function__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        name = func.__name__ if hasattr(func, '__name__') else str(func)
        self.counts[name] += 1
        return func(*args, **kwargs)

# Run a model forward pass with counting
model = nn.Sequential(
    nn.Linear(32, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

x = torch.randn(8, 32)

with OpCounter() as counter:
    output = model(x)

print("Operation counts during forward pass:")
for op, count in sorted(counter.counts.items(), key=lambda x: -x[1]):
    print(f"  {op:20s}: {count}")
print(f"\nTotal ops: {sum(counter.counts.values())}")

## __torch_function__ vs __torch_dispatch__ — Key Differences

| Aspect | `__torch_function__` | `__torch_dispatch__` |
|--------|---------------------|---------------------|
| Level | Python API (torch.add, F.relu) | ATen operators (aten::add.Tensor) |
| Sees | High-level operations | Decomposed low-level ops |
| Before/After autograd | Before | After |
| F.relu becomes | One call: `relu` | `aten::relu` or `aten::clamp_min` |
| Use case | Custom tensor semantics | Backend implementations, tracing |
| Subclass | `torch.Tensor` subclass | `torch.Tensor` subclass |
| Mode class | `TorchFunctionMode` | `TorchDispatchMode` |

The key insight: `__torch_dispatch__` sees the **decomposed ATen ops** that the backend actually executes, while `__torch_function__` sees the user-facing API.

## TorchDispatchMode: ATen-Level Operation Counter

Let's see what the **ATen level** sees for the same operations. This is "below" autograd — it sees the primitive ops.

In [ ]:
class ATenCounter(TorchDispatchMode):
    """Count ATen-level operations."""

    def __init__(self):
        super().__init__()
        self.counts = defaultdict(int)

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        self.counts[str(func)] += 1
        return func(*args, **kwargs)

# Same model, same input — but now we see ATen ops
x = torch.randn(8, 32)

with ATenCounter() as aten_counter:
    output = model(x)

print("ATen-level operations during forward pass:")
for op, count in sorted(aten_counter.counts.items(), key=lambda x: -x[1])[:15]:
    print(f"  {op:45s}: {count}")
print(f"\nTotal ATen ops: {sum(aten_counter.counts.values())}")
print(f"\n(Compare: TorchFunctionMode saw {sum(counter.counts.values())} Python-level ops)")

## Side-by-Side: What Each Level Sees for F.relu

Let's compare what `__torch_function__` vs `__torch_dispatch__` observe for the exact same code.

In [ ]:
class VerboseFunctionMode(TorchFunctionMode):
    def __init__(self):
        super().__init__()
        self.ops = []
    def __torch_function__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        name = f"{func.__module__}.{func.__name__}" if hasattr(func, '__module__') else str(func)
        self.ops.append(name)
        return func(*args, **kwargs)

class VerboseDispatchMode(TorchDispatchMode):
    def __init__(self):
        super().__init__()
        self.ops = []
    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        self.ops.append(str(func))
        return func(*args, **kwargs)

x = torch.randn(4, 4)

# What does __torch_function__ see?
with VerboseFunctionMode() as tf_mode:
    y = F.relu(F.linear(x, torch.randn(4, 4), torch.randn(4)))

# What does __torch_dispatch__ see?
with VerboseDispatchMode() as td_mode:
    y = F.relu(F.linear(x, torch.randn(4, 4), torch.randn(4)))

print("__torch_function__ sees (Python API level):")
for op in tf_mode.ops:
    print(f"  {op}")

print(f"\n__torch_dispatch__ sees (ATen level):")
for op in td_mode.ops:
    print(f"  {op}")

## Practical Example: UnitTensor (Physical Units)

A powerful use of `__torch_function__`: tensors with physical units that enforce dimensional consistency (can't add meters to seconds).

In [ ]:
class UnitTensor(torch.Tensor):
    """Tensor with physical units. Enforces unit consistency on add/sub."""

    @staticmethod
    def __new__(cls, data, unit=""):
        return torch.Tensor._make_subclass(cls, data if isinstance(data, torch.Tensor) else torch.tensor(data, dtype=torch.float))

    def __init__(self, data, unit=""):
        self._unit = unit

    @property
    def unit(self):
        return self._unit

    @classmethod
    def __torch_function__(cls, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}

        # Extract units from args
        units = [a._unit for a in torch.utils._pytree.tree_leaves(args) if isinstance(a, UnitTensor)]

        # For add/sub: units must match
        if func in (torch.add, torch.Tensor.add, torch.sub, torch.Tensor.sub, torch.Tensor.__add__, torch.Tensor.__sub__):
            if len(units) >= 2 and units[0] != units[1]:
                raise TypeError(f"Unit mismatch: cannot {func.__name__} '{units[0]}' and '{units[1]}'")

        # Unwrap, compute, rewrap
        def unwrap(t):
            return torch.Tensor._make_subclass(torch.Tensor, t) if isinstance(t, UnitTensor) else t

        raw_args = torch.utils._pytree.tree_map(unwrap, args)
        raw_kwargs = torch.utils._pytree.tree_map(unwrap, kwargs)
        result = func(*raw_args, **raw_kwargs)

        if isinstance(result, torch.Tensor):
            # Determine output unit
            if func in (torch.mul, torch.Tensor.mul, torch.Tensor.__mul__):
                out_unit = "*".join(units) if len(units) == 2 else (units[0] if units else "")
            elif func in (torch.div, torch.Tensor.div, torch.Tensor.__truediv__):
                out_unit = f"{units[0]}/{units[1]}" if len(units) == 2 else (units[0] if units else "")
            else:
                out_unit = units[0] if units else ""
            return UnitTensor(result, unit=out_unit)
        return result

    def __repr__(self):
        data = torch.Tensor._make_subclass(torch.Tensor, self)
        return f"UnitTensor({data.tolist()}, unit='{self._unit}')"

# Create tensors with units
distance = UnitTensor([10.0, 20.0, 30.0], unit="m")
time_s = UnitTensor([2.0, 4.0, 6.0], unit="s")
distance2 = UnitTensor([5.0, 10.0, 15.0], unit="m")

print(f"distance = {distance}")
print(f"time = {time_s}")

# Valid: add meters to meters
total_distance = distance + distance2
print(f"\ndistance + distance2 = {total_distance}")

# Valid: divide meters by seconds → velocity
velocity = distance / time_s
print(f"distance / time = {velocity}")

# Invalid: add meters to seconds
try:
    bad = distance + time_s
except TypeError as e:
    print(f"\ndistance + time → ERROR: {e}")

## TorchDispatchMode for Profiling: FLOP Counter

`TorchDispatchMode` is ideal for counting FLOPs because it sees the actual compute ops (matmul, add) at the ATen level.

In [ ]:
class SimpleFLOPCounter(TorchDispatchMode):
    """Estimate FLOPs by counting matmul/addmm operations."""

    def __init__(self):
        super().__init__()
        self.flops = 0
        self.op_flops = {}

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        result = func(*args, **kwargs)

        # Count FLOPs for key operations
        op_name = str(func)
        if "addmm" in op_name or "mm" in op_name:
            # Matrix multiply: 2 * M * N * K FLOPs
            if len(args) >= 2:
                mat_args = [a for a in args if isinstance(a, torch.Tensor) and a.dim() == 2]
                if len(mat_args) >= 2:
                    m, k = mat_args[-2].shape
                    _, n = mat_args[-1].shape
                    flops = 2 * m * k * n
                    self.flops += flops
                    self.op_flops[op_name] = self.op_flops.get(op_name, 0) + flops

        return result

# Count FLOPs in a model forward pass
model = nn.Sequential(
    nn.Linear(128, 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 10),
)
x = torch.randn(32, 128)

with SimpleFLOPCounter() as flop_counter:
    output = model(x)

print(f"Estimated FLOPs for forward pass:")
print(f"  Total: {flop_counter.flops:,}")
for op, flops in flop_counter.op_flops.items():
    print(f"  {op}: {flops:,}")
print(f"\n(Batch=32, so per-sample ≈ {flop_counter.flops // 32:,} FLOPs)")

## Dispatch Stack Diagram

The full dispatch order when you call an operation:

```
torch.add(a, b)                    ← User code
    │
    ▼
[TorchFunctionMode stack]          ← Modes applied in LIFO order
    │
    ▼
[__torch_function__]               ← If 'a' or 'b' is a Tensor subclass
    │
    ▼
[Autograd]                         ← Records for backward (if requires_grad)
    │
    ▼
[TorchDispatchMode stack]          ← Modes applied in LIFO order
    │
    ▼
[__torch_dispatch__]               ← If 'a' or 'b' has dispatch key
    │
    ▼
[Backend kernel]                   ← CPU/CUDA/MPS actual computation
```

**Key rules:**
- Modes stack (LIFO): the last `__enter__`'d mode runs first
- `__torch_function__` takes priority over `__torch_dispatch__`
- Autograd sits between the two levels
- You can nest modes of the same type

## Nesting Modes: Multiple Modes Active Simultaneously

Modes compose — you can have a logging mode AND a validation mode active at the same time.

In [ ]:
class NaNChecker(TorchDispatchMode):
    """Check for NaN values after every operation."""

    def __init__(self):
        super().__init__()
        self.nan_ops = []

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        result = func(*args, **kwargs)
        if isinstance(result, torch.Tensor) and result.is_floating_point():
            if torch.isnan(result).any():
                self.nan_ops.append(str(func))
        return result

class ShapeTracker(TorchDispatchMode):
    """Track output shapes of all operations."""

    def __init__(self):
        super().__init__()
        self.shapes = []

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        result = func(*args, **kwargs)
        if isinstance(result, torch.Tensor):
            self.shapes.append((str(func), result.shape))
        return result

# Nest both modes
x = torch.randn(4, 4)
x[0, 0] = float('nan')  # introduce a NaN

with NaNChecker() as nan_mode:
    with ShapeTracker() as shape_mode:
        y = x * 2
        z = y + 1
        w = z.sum()

print("NaN detected in ops:", nan_mode.nan_ops)
print(f"\nShape tracking:")
for op, shape in shape_mode.shapes:
    print(f"  {op:40s} → {shape}")

## Practical: TorchFunctionMode for Dtype Validation

A mode that validates all tensors entering operations are float32 — useful for catching accidental mixed-precision bugs.

In [ ]:
class Float32Validator(TorchFunctionMode):
    """Warns if any non-float32 floating-point tensor is used in an operation."""

    def __init__(self):
        super().__init__()
        self.violations = []

    def __torch_function__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        # Check all tensor args
        for arg in torch.utils._pytree.tree_leaves(args):
            if isinstance(arg, torch.Tensor) and arg.is_floating_point() and arg.dtype != torch.float32:
                self.violations.append((func.__name__ if hasattr(func, '__name__') else str(func), arg.dtype))
        return func(*args, **kwargs)

# Test with valid float32
with Float32Validator() as validator:
    a = torch.randn(4)  # float32 by default
    b = torch.randn(4)
    c = a + b

print(f"All float32: violations = {len(validator.violations)}")

# Test with accidental float16
with Float32Validator() as validator:
    a = torch.randn(4)
    b = torch.randn(4).half()  # float16!
    c = a + b.float()  # even after cast, b entered as float16

print(f"Mixed dtypes: violations = {len(validator.violations)}")
for func_name, dtype in validator.violations[:3]:
    print(f"  {func_name} received {dtype}")

## When to Use Which Protocol

| Goal | Use |
|------|-----|
| Custom tensor semantics (quantized, sparse, scaled) | `__torch_function__` subclass |
| Log/profile operations in a scope | `TorchFunctionMode` |
| Build a new backend or device | `__torch_dispatch__` subclass |
| Count ATen ops / estimate FLOPs | `TorchDispatchMode` |
| Validate tensor properties | `TorchFunctionMode` |
| Fake/meta tensors (shape-only computation) | `__torch_dispatch__` subclass |
| Trace execution for compilation | `TorchDispatchMode` |

## Try It Yourself

**Exercise:** Create a `TorchFunctionMode` that validates all tensors in operations are `float32`. If any tensor has a different floating-point dtype, raise a `TypeError`.

Test it with:
1. Two float32 tensors (should work)
2. A float32 + float64 operation (should raise)
3. An integer tensor (should be allowed — only check floating point)

In [ ]:
# YOUR CODE HERE
class StrictFloat32Mode(TorchFunctionMode):
    """Raise TypeError if any floating-point tensor is not float32."""

    def __torch_function__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        # Check all tensor arguments
        for arg in torch.utils._pytree.tree_leaves(args):
            if isinstance(arg, torch.Tensor) and arg.is_floating_point() and arg.dtype != torch.float32:
                raise TypeError(
                    f"{func.__name__}: expected float32, got {arg.dtype}"
                )
        return func(*args, **kwargs)

# Test 1: float32 + float32 (should work)
with StrictFloat32Mode():
    a = torch.randn(4)  # float32
    b = torch.randn(4)  # float32
    c = a + b
    print(f"Test 1 passed: float32 + float32 = {c.dtype}")

# Test 2: float32 + float64 (should raise)
try:
    with StrictFloat32Mode():
        a = torch.randn(4)
        b = torch.randn(4, dtype=torch.float64)
        c = a + b
except TypeError as e:
    print(f"Test 2 passed: caught error — {e}")

# Test 3: integer tensors (should be allowed)
with StrictFloat32Mode():
    a = torch.tensor([1, 2, 3])  # int64
    b = torch.tensor([4, 5, 6])  # int64
    c = a + b
    print(f"Test 3 passed: int64 allowed — {c}")

## Key Takeaways

1. **`__torch_function__`** intercepts at the Python API level — sees `torch.add`, `F.relu`, etc.
2. **`__torch_dispatch__`** intercepts at the ATen level — sees decomposed ops like `aten::add.Tensor`
3. **`TorchFunctionMode`** — context manager for scoped `__torch_function__` without tensor subclassing
4. **`TorchDispatchMode`** — context manager for scoped `__torch_dispatch__` interception
5. **Tensor subclasses** use `__torch_function__` to define custom semantics (ScaledTensor, UnitTensor)
6. **Modes stack** in LIFO order — you can nest multiple modes simultaneously
7. **`__torch_function__` runs before autograd**, `__torch_dispatch__` runs after
8. **Use `torch_function` for** custom tensor types, validation, and high-level logging
9. **Use `torch_dispatch` for** FLOP counting, backend implementations, and ATen-level tracing
10. **PyTorch internals use these extensively** — FakeTensor, ProxyTensor, DTensor all use `__torch_dispatch__`